<h1>Atividade 4 -- Tópicos para Computação 1 -- 2026.1</h1>

- Escola Superior de Tecnologia
- Profa. Dra. Elloá B. Guedes (ebgcosta@uea.edu.br)
- www.elloaguedes.com
- github.com/elloa
- Data: 19 de março de 2026

<h3>Descrição</h3>
A atividade consiste em construir uma rede neural convolucional usando transfer learning da rede Inception V3 para classificar o Stanford Dogs Dataset, um dataset com imagens de 120 raças de cachorro de todo o mundo e mais de 20 mil imagens para treino e teste

<h3>Requisitos:</h3>

- Uso de Transfer Learning
- Uso de Taxa de Aprendizado Adaptativa
- Data Augmentation
- L2 regularizatiom
- Early Stopping
- Gráficos de loss/acurácia no treino e validação
- Métricas no conjunto de teste
- Elaboração de um conjunto de slides com os resultados dos experimentos e apresentação da arquitetura
- Monitoramento do tempo de treinamento
- Quantidade de parâmetros totais e ajustáveis

<h3>Equipe:</h3>

- Ana Flavia de Castro Segadillha da Silva
- Davi Aguiar Moreira
- Guilherme Goncalves Moraes
- Luiz Fernando Borges Brito
- Pedro Vitor Barros Maranhao

<h2>Bibliotecas</h2>

In [2]:
import os
import kagglehub
import zipfile
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from torchvision.models import inception_v3, Inception_V3_Weights
import random

In [3]:
print(torch.cuda.is_available())

False


<h2>Extração e filtragem do dataset</h2>

In [3]:
#Opção 1 importar do kagglehub
path = kagglehub.dataset_download("miljan/stanford-dogs-dataset-traintest")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\pedro\.cache\kagglehub\datasets\miljan\stanford-dogs-dataset-traintest\versions\1


In [ ]:
#Opção 2 pegar da pasta opt

zip_path = "/opt/topicos/stanford-dogs.zip"
#Não sei se colocar na minha pasta home permite que todos rodem
extract_path = "/home/pvbm.eng23/dataset"  

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

pastas = os.listdir(extract_path)
path=extract_path

print(pastas)

In [4]:
#Criação de novos paths para acessar os dados de treino e teste

novo_path = os.path.join(path, "cropped")

test_path = os.path.join(novo_path, "test")

train_path = os.path.join(novo_path, "train")

In [7]:
#Função para retirar "n[id]" do início das pastas buscando por "-"
def tirar_id(path):
    for item in os.listdir(path):
        nome_bruto = os.path.join(path, item)
        segmentado = item.split("-", 1)     
        novo_nome = segmentado[1]
        nome_limpo = os.path.join(path, novo_nome)   
        os.rename(nome_bruto, nome_limpo)

tirar_id(test_path)
tirar_id(train_path)

<h2>Transformações com Data Augmentation</h2>

In [8]:
train_transform = transforms.Compose([transforms.Resize((256, 256)),transforms.RandomCrop(224), transforms.RandomHorizontalFlip(),
                                      transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1), ## varia brilho, contraste, saturação e matiz
                                      transforms.RandomRotation(10), transforms.ToTensor(),
                                      transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])])
## obs: o conjunto de teste não recebe o data augmentation, só a normalização.

test_transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(),
                                     transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])])

train_dataset = ImageFolder(root=train_path, transform=train_transform)
test_dataset  = ImageFolder(root=test_path,  transform=test_transform)

train_load = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_load  = DataLoader(test_dataset,  batch_size=1,  shuffle=False)

images, labels = next(iter(train_load))
print("Batch shape:", images.shape, labels.shape)

Batch shape: torch.Size([32, 3, 224, 224]) torch.Size([32])


<h2>Inception V3</h2>

In [22]:
model = inception_v3(weights=Inception_V3_Weights.DEFAULT, aux_logits=True)
classes = 120
#Camada de saída principal e auxiliar
model.fc = nn.Linear(model.fc.in_features, classes)
model.AuxLogits.fc = nn.Linear(model.AuxLogits.fc.in_features, classes)

In [23]:
for param in model.parameters():
    param.requires_grad = False

# liberar só a camada de saída para treino
for param in model.fc.parameters():
    param.requires_grad = True

In [24]:
print(model)

Inception3(
  (Conv2d_1a_3x3): BasicConv2d(
    (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_2a_3x3): BasicConv2d(
    (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_2b_3x3): BasicConv2d(
    (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (maxpool1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (Conv2d_3b_1x1): BasicConv2d(
    (conv): Conv2d(64, 80, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(80, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_4a_3x3): BasicConv2d(
    (conv): Conv2d(80, 192, kernel_size=(3, 3), stri

<h2>Configurações do treino</h2>

<h2>Métricas do treino</h2>

<h2>Avaliação do Modelo</h2>

<h2>Conclusões</h2>